## 6.3 Per Fold

In this section, we evaluate directional classification with **Logistic Regression** under strict **walk-forward validation**.  
The analysis is executed with **5 temporal folds** (`n_splits = 5`) to examine temporal robustness rather than single-split performance.

In [ ]:
fs_fold_df = src.models.run_forward_selection_per_fold(feature_data, n_splits=src.config.DEF_SPLITS)
src.models.plot_forward_selection_per_fold(fs_fold_df, feature_data, n_splits=src.config.DEF_SPLITS)


### 6.3.1 Five-Fold Accuracy and Regime Context

This subsection reports fold-level diagnostics per ticker to explain temporal variability in out-of-sample accuracy.  
The diagnostics integrate: validation accuracy, explicit train/validation intervals, market-regime composition (`Regime_Bull`, MA200-based), regime-shift magnitude, and validation-period return volatility.  
The objective is to connect model behavior to market conditions while preserving methodological reproducibility across reruns.


In [ ]:
fold_context_df, fold_context_summary_df = src.models.run_fold_regime_context_analysis(
    feature_data,
    candidate_features_map=globals().get("top_experiment_features", {}),
    n_splits=src.config.DEF_SPLITS,
)

src.models.display_fold_regime_context_tables(fold_context_df, fold_context_summary_df)
src.models.plot_fold_regime_context(fold_context_df)


### 6.3.2 Interpretive Layer: Linking Fold Variability to Market Conditions

Fold-level dispersion is expected in financial time series because each fold is validated on a different market segment.
Interpretation should therefore prioritize structural drivers over isolated point estimates.

Core interpretation dimensions:

- **Regime continuity vs. regime shift**: folds typically generalize better when training and validation share similar market-state composition.
- **Shock intensity**: abrupt macro shocks increase noise and usually weaken short-horizon directional predictability.
- **Volatility and repricing speed**: rapid repricing reduces the persistence of historical feature-target relationships.

#### Macro Context (2020-2025)

| Period (approx.) | Market characterization | Typical implication for folds |
|---|---|---|
| Early 2020 | COVID shock and deleveraging | Higher instability; weaker and less consistent folds are more likely. |
| Mid 2020-2021 | Reopening and liquidity support | Stronger trend persistence; better generalization is more likely. |
| 2022 | Inflation shock and aggressive rate hikes | Rotation and repricing; larger fold dispersion is common. |
| 2023-2024 | AI-led mega-cap leadership | Favorable for selected leaders, with potentially higher cross-ticker asymmetry. |
| 2025 | Mixed late-cycle signals | Transition risk increases; out-of-sample stability may soften. |

#### Ticker-Specific Considerations (General)

**AAPL**: often relatively resilient, yet sensitive to demand-cycle and global supply-demand uncertainty.  
**MSFT**: usually more stable, but exposed to broad valuation compression in rate-shock environments.  
**AMZN**: typically more cyclical and risk-sentiment sensitive, with potentially wider fold dispersion.  
**GOOG**: strongly influenced by ad-cycle visibility, regulatory headlines, and AI-competition narratives.

#### Synthesis

Fold outcomes should be interpreted as evidence of temporal robustness under changing market regimes, not as fixed performance rankings.
